# Análisis exploratorio de anuncios de vehículos usados

En este cuaderno revisamos el conjunto de datos "vehicles_us.csv" para preparar el panel interactivo del Sprint 7. Nos enfocamos en conocer las variables, detectar valores faltantes y construir dos gráficos con Plotly. El objetivo principal del sprint es desplegar una aplicación web, por lo que mantenemos un análisis exploratorio breve y verificable.

## 1. Cargamos las librerías y el archivo

Utilizamos pandas para explorar los datos y Plotly para preparar visualizaciones interactivas. Buscamos el archivo CSV en la raíz del proyecto para que el notebook también funcione después de clonar el repositorio.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

candidates = [Path("vehicles_us.csv"), Path("../vehicles_us.csv")]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        "Descarga vehicles_us.csv de TripleTen y colócalo en la raíz del proyecto."
    )

df = pd.read_csv(data_path)
print(f"Registros: {len(df)}")
print(f"Columnas: {len(df.columns)}")
display(df.head())

## 2. Comprobamos la estructura y los valores faltantes

Revisamos los tipos de datos y contamos los valores ausentes. No reemplazamos información sin evidencia, ya que eso podría modificar la distribución observada.

In [ ]:
print("Tipos de datos:")
df.info()

missing = df.isna().sum().sort_values(ascending=False)
display(missing.rename("Valores faltantes").to_frame())

## 3. Estadísticas descriptivas

Primero examinamos el precio, el kilometraje y el año del modelo. Calculamos la mediana porque los valores muy altos pueden afectar considerablemente a la media.

In [ ]:
numerical_columns = ["price", "odometer", "model_year"]
for col in numerical_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

display(df[numerical_columns].describe().T)
print("Precio mediano:", df["price"].median())
print("Kilometraje mediano:", df["odometer"].median())

## 4. Distribución del kilometraje

Utilizamos un histograma para observar la concentración de anuncios por kilometraje. Prestamos atención al intervalo con mayor frecuencia, al grado de dispersión y a los valores extremos. El gráfico se calcula únicamente con observaciones de kilometraje válidas.

In [ ]:
mileage = df.loc[df["odometer"].notna() & df["odometer"].ge(0)]
fig_hist = px.histogram(
    mileage,
    x="odometer",
    nbins=55,
    title="Distribución del kilometraje de los vehículos",
    labels={"odometer": "Odómetro (millas)"},
)
fig_hist.update_layout(yaxis_title="Número de anuncios")
fig_hist.show()

## 5. Relación entre kilometraje y precio

Dibujamos un gráfico de dispersión para identificar cómo varían los precios según el kilometraje. La condición del vehículo puede explicar parte de la variabilidad. Limitamos los precios al percentil 99 para que los valores extremos no oculten el patrón principal, e indicamos explícitamente esta decisión. No interpretamos una asociación visual como causalidad.

In [ ]:
chart_df = df.dropna(subset=["price", "odometer", "condition"]).copy()
chart_df = chart_df.loc[chart_df["price"].ge(0) & chart_df["odometer"].ge(0)]
price_cap = df["price"].quantile(0.99)
chart_df = chart_df.loc[chart_df["price"].le(price_cap)]

if len(chart_df) > 12000:
    chart_df = chart_df.sample(12000, random_state=42)

fig_scatter = px.scatter(
    chart_df,
    x="odometer", y="price", color="condition", opacity=0.5,
    render_mode="webgl",
    title="Precio anunciado en función del kilometraje",
    labels={
        "odometer": "Odómetro (millas)",
        "price": "Precio (USD)",
        "condition": "Estado",
    },
)
fig_scatter.show()
print(f"Límite de precio para este gráfico (percentil 99): {price_cap:.2f}")

## 6. Comprobamos las categorías

Conocemos las frecuencias de los tipos y estados de vehículos para preparar filtros comprensibles dentro de Streamlit.

In [ ]:
display(df["type"].value_counts(dropna=False).rename("Anuncios").to_frame())
display(df["condition"].value_counts(dropna=False).rename("Anuncios").to_frame())

## 7. Conclusión

En este análisis comprobamos la estructura del archivo, los valores faltantes y las distribuciones necesarias para crear el panel web. El histograma permite explorar el kilometraje y el gráfico de dispersión muestra la relación entre precio, kilometraje y condición del vehículo.

El archivo contiene 51 525 anuncios y 13 columnas. El precio mediano es de 9 000 USD y el odómetro mediano, calculado con los registros disponibles, es de 113 000 millas. Hay 7 892 anuncios sin odómetro y 3 619 sin año del modelo. Esta falta de información reduce la cobertura de los gráficos que usan el odómetro y de las selecciones que acotan el rango de años.